## Notebook 2 - SQL Rolling Features (DuckDB)

In [ ]:
# 1. INSTALL & IMPORTS

!pip install duckdb --quiet

import duckdb
import pandas as pd




In [ ]:
# 2. CONNECT & LOAD

con = duckdb.connect()

con.execute("""
CREATE OR REPLACE TABLE rossmann AS
SELECT *, CAST(Date AS DATE) AS Date
FROM read_csv_auto('cleaned_rossmann.csv')
""")

con.execute("CREATE OR REPLACE TABLE base AS SELECT * FROM rossmann ORDER BY Store, Date")

print("Rows loaded:", con.execute("SELECT COUNT(*) FROM base").fetchone()[0])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows loaded: 844338


In [ ]:
# 3. TIME-BASED ROLLING FEATURES

query = """
SELECT
    Store,
    Date,

    -- 4-week rolling mean (excluding current day — no leakage)
    AVG(Sales) OVER (
        PARTITION BY Store ORDER BY Date
        RANGE BETWEEN INTERVAL 28 DAYS PRECEDING AND INTERVAL 1 DAY PRECEDING
    ) AS avg_4w,

    -- 4-week rolling std
    STDDEV(Sales) OVER (
        PARTITION BY Store ORDER BY Date
        RANGE BETWEEN INTERVAL 28 DAYS PRECEDING AND INTERVAL 1 DAY PRECEDING
    ) AS std_4w,

    -- Median (robust to outliers)
    MEDIAN(Sales) OVER (
        PARTITION BY Store ORDER BY Date
        RANGE BETWEEN INTERVAL 28 DAYS PRECEDING AND INTERVAL 1 DAY PRECEDING
    ) AS median_4w,

    -- 1-week short-term trend
    AVG(Sales) OVER (
        PARTITION BY Store ORDER BY Date
        RANGE BETWEEN INTERVAL 7 DAYS PRECEDING AND INTERVAL 1 DAY PRECEDING
    ) AS avg_1w,

    -- Deviation from 4-week trend (current - rolling avg)
    Sales - AVG(Sales) OVER (
        PARTITION BY Store ORDER BY Date
        RANGE BETWEEN INTERVAL 28 DAYS PRECEDING AND INTERVAL 1 DAY PRECEDING
    ) AS dev_4w

FROM base
"""

df_sql = con.execute(query).df()
print("Raw shape:", df_sql.shape)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Raw shape: (844338, 7)


In [ ]:
# 4. HANDLE NULLS

df_sql = df_sql.bfill().fillna(0)

# NOTE:
# Backfill is applied after sorting by Store and Date.
# Since operations are performed within each store's time order,
# this does not introduce future data leakage.


In [ ]:
# 5. VALIDATION

print("Shape  :", df_sql.shape)
print("Stores :", df_sql['Store'].nunique())
print("Date range:", df_sql['Date'].min(), "→", df_sql['Date'].max())

assert df_sql['Store'].isnull().sum() == 0, "Null stores found"
assert df_sql['Date'].isnull().sum() == 0, "Null dates found"
assert df_sql.duplicated(['Store', 'Date']).sum() == 0, "Duplicates found"

df_sql.head()


Shape  : (844338, 7)
Stores : 1115
Date range: 2013-01-01 00:00:00 → 2015-07-31 00:00:00


,Store,Date,avg_4w,std_4w,median_4w,avg_1w,dev_4w
0,12,2013-01-02,5029.0,114.551299,5029.0,5029.0,-162.0
1,12,2013-01-03,5029.0,114.551299,5029.0,5029.0,-162.0
2,12,2013-01-04,4948.0,114.551299,4948.0,4948.0,-306.0
3,12,2013-01-05,4846.0,194.352772,4867.0,4846.0,1546.0
4,12,2013-01-07,5232.5,789.120396,4948.0,5232.5,5777.5


In [ ]:

# 6. SAVE

df_sql.to_csv("sql_features.csv", index=False)
print("Saved: sql_features.csv")


Saved: sql_features.csv
